# ARGUS-FEDER — Answering GA's disruption competency questions

General Atomics maintains a set of 27 competency questions for DIII-D data,
split into an ELM half and a disruption half. This notebook works through the
**disruption half, Q14–Q27**, against a stored disruption index built from
`disruption-py` output.

Each question is tagged with what the index can actually deliver: answerable
now, answerable for one pipeline only, or correctly refused.

## The one thing to keep in mind

GA's questions are written for **three** disruption pipelines. We have run
**one** — `disruption-py`, the public MIT-PSFC code. Questions that ask for
agreement *across* pipelines cannot be answered yet, and the correct response
is to say which pipeline was used and what is missing, not to answer as though
one pipeline were all of them.

The index also holds **20 shots, balanced 10 disrupted / 10 not**. It is a
demonstration slice, not a sample of the archive. Any question phrased as a
rate or a fraction is unanswerable from it by construction.

In [ ]:
%reset

---
# A. Answerable now, from this index

## Q20 — Which pipeline produced a label, at what version, from which inputs?

*provenance · **answerable***

Provenance is answerable by construction once a pipeline is run and what was
run is recorded alongside the labels it produced.

In [2]:
%%ask --review
For the disruption labels in the index: which code produced them, at what
version and commit, when was it run and by whom, and what were its inputs? Show
me the full provenance record.

I'll start by planning this provenance lookup, then read the relevant skill documentation.

Reading the disruption skill instructions plus its existing learnings file in parallel.

The skill documents `index_info()` as the provenance call, and the Learnings file flags an expired-token trap. Writing a query script that applies the learned fix.

Now running the script to fetch the provenance record from the index.

The provenance record is in. Since the question is specifically about the *labels*, I'll also document the two label sources and where they disagree.

Running the label-sources script to capture the label lineage alongside the code provenance.

**Full provenance record for the disruption index** (from `index_info()`, which reads it straight from the source file's global attributes):

**Producing code**
- **Package** : `disruption_py` (the MIT-PSFC disruption-py pipeline)
- **Version** : `0.15.0.dev0`
- **Commit** : `9987784e78ef9736e2fe42142683b3458ff38d8d`
- **Source repo** : https://github.com/MIT-PSFC/disruption-py/tree/9987784e78ef9736e2fe42142683b3458ff38d8d
- **Tokamak** : D3D

**Run details**
- **Run by** : `sammuli`
- **Run on** : `omega06.gat.com`
- **Produced at** : 2026-08-19T07:36:50.648611 (no timezone in the file's attribute; naive timestamp)
- **Source file produced** : `d3d_disruption_example.nc`
- **Ingested into this index** : 2026-08-19T20:46:49+00:00 (run_id 1)

**Inputs**
- The pipeline's inputs are DIII-D shot data for the slice covered: shots **194000–199687** , of which **20 shots** were ingested (10 disrupted, 3778 sample rows).
- For the labels themselves, provenance splits into two distinct inputs:
  1. **The label** — `time_until_disrupt`, taken from the **human-curated `disruptions` table in d3drdb** (the DIII-D relational database). This is the authoritative label `disruption_label()` returns.
  2. **A derived indicator, not the label** — `current_quench_time`, disruption-py's own estimate from the Ip decay at commit `9987784...`. It **disagrees with the curated label on 3 of the 20 shots** (198244, 198643, 199062 — all curated-as-not-disrupted but Ip-decay-flagged), so the two are kept deliberately separate.

**Caveats worth carrying with this record**
- The 20-shot set is a **constructed slice, not a random sample** of DIII-D (`is_representative_sample: False`) — it was chosen to span a range of behaviour with a deliberate 10/10 disrupted/non-disrupted split. No archive-wide disruption rate can be computed from it.
- The parameter list and the known parameter defects are properties of this specific revision (`0.15.0.dev0` @ `9987784`); a re-ingest from a newer disruption-py would change both.

The query scripts are saved as `provenance_query.py` and `label_sources.py` in the working directory.

## Q15 — Which quantities are disruption precursors, and what measures each?

*precursor map · **answerable***

The assessment noted that `disruption-py`'s feature set *is* the precursor
map — n=1 mode amplitude, betap, li, q95, radiated fraction, locked mode,
v_loop. Now that it has been run, that map is queryable with units attached.

In [3]:
%%ask --review
Which quantities in this dataset are disruption precursors, and what does
each one measure? Group them by the physics method that produces them, and give
the units for each.

Reading the helper module to confirm the exact function signatures and return shapes.

Skill and learnings read — the token-env fix from Learnings goes at the top of my script. Now querying the parameter dictionary grouped by method.

All 63 parameters retrieved with their method groups, descriptions, units, and caveats. Now I can classify the precursor set versus the label layer.

The 63 columns in this index split into two layers: **60 physics quantities** (the precursor candidates a warning system would consume) and **3 label/phase columns** that describe the shot's outcome, not its state. Everything below is read from the index's own parameter dictionary (`parameters()` / `parameter_info()`), produced by disruption-py `0.15.0.dev0` @ `9987784e78ef9736e2fe42142683b3458ff38d8d` — the parameter list and its defects are properties of that revision.

Units convention: `1` means genuinely dimensionless; `instrument` (h_alpha) means uncalibrated detector units, not SI.

## Precursor quantities, grouped by producing physics method

**Equilibrium reconstruction — `get_efit_parameters`** (filled only where EFIT converged; gaps concentrate at shot end, 84.6% fill)

| Parameter | Measures | Units |
|---|---|---|
| q95 | Safety factor at 95% flux surface — distance from the low-q kink/tearing boundary (disruptive limit: approaching 2 from above) | 1 |
| q0 | Safety factor on axis | 1 |
| qstar | Cylindrical safety factor | 1 |
| wmhd | Stored magnetic energy — confinement quality | J |
| dwmhd_dt | Rate of change of stored energy | W |
| beta_n | Normalised beta | % m T MA⁻¹ |
| beta_p | Poloidal beta — normalized plasma pressure | 1 |
| dbetap_dt | Rate of change of poloidal beta | s⁻¹ |
| li | Internal inductance — current-profile peaking | 1 |
| dli_dt | Rate of change of internal inductance | s⁻¹ |
| kappa | Elongation | 1 |
| chisq | EFIT fit quality (reconstruction reliability) | 1 |
| upper_gap / lower_gap | Plasma-to-wall gap, top / bottom — proximity of the plasma to limiting surfaces | m |

**MHD modes — `get_n1_bradial_parameters` / `get_n1rms_parameters`** (98.8–100% fill)

| Parameter | Measures | Units |
|---|---|---|
| n1rms | n=1 RMS magnetic amplitude — locked/unstable mode growth, the classic locked-mode precursor | T |
| n1rms_normalized | n=1 RMS normalised by toroidal field | 1 |
| n_equal_1_mode | n=1 radial field amplitude | T |
| n_equal_1_normalized | n=1 amplitude normalised by toroidal field | 1 |

**Density — `get_density_parameters`** (line-averaged density)

| Parameter | Measures | Units |
|---|---|---|
| n_e | Line-averaged electron density | m⁻³ |
| greenwald_fraction | n_e against the Greenwald density limit — the density-limit disruption metric | 1 |
| dn_dt | Rate of change of density | m⁻³ s⁻¹ |

**Plasma current — `get_ip_parameters`**

| Parameter | Measures | Units |
|---|---|---|
| ip | Measured plasma current — its decay is the current quench | A |
| dip_dt | Rate of change of plasma current | A s⁻¹ |
| ip_prog | Programmed (commanded) plasma current — reference for tracking | MA |

**Ohmic drive — `get_ohmic_parameters`**

| Parameter | Measures | Units |
|---|---|---|
| v_loop | Loop voltage — resistive state of the plasma | V |
| p_ohm | Ohmic heating power | W |

**Heating and radiation — `get_power_parameters`**

| Parameter | Measures | Units |
|---|---|---|
| p_rad | Radiated power (bolometry, causal 10 ms smoothing) — radiative-collapse precursor | W |
| radiated_fraction | p_rad / total input power — impurity/dilution indicator | 1 |
| p_nbi | Neutral-beam injected power | W |
| p_ech | Electron-cyclotron heating power | W |

**Profile peaking, real-time — `get_peaking_factors`** (core-vs-all and divertor asymmetries; precursor to core radiative collapse and divertor detachment)

| Parameter | Measures | Units |
|---|---|---|
| prad_peaking_cva_rt | Radiated-power peaking, core vs. all | 1 |
| prad_peaking_xdiv_rt | Radiated-power peaking, divertor region | 1 |
| ne_peaking_cva_rt | Density peaking, core vs. all | 1 |
| te_peaking_cva_rt | Temperature peaking, core vs. all | 1 |

**Real-time twins** — a parallel measurement path computed by the plasma-control system rather than offline analysis, one per group above:

| Method | Parameters | Units |
|---|---|---|
| get_rt_efit_parameters (real-time equilibrium) | q95_rt, wmhd_rt, beta_p_rt, li_rt, chisq_rt | 1, J, 1, 1, 1 |
| get_rt_ip_parameters (real-time current) | ip_rt, ip_prog_rt, dip_dt_rt, dipprog_dt_rt | A, A, A s⁻¹, A s⁻¹ |
| get_rt_density_parameters (real-time density) | n_e_rt, greenwald_fraction_rt, dn_dt_rt | m⁻³, 1, m⁻³ s⁻¹ |

**Vertical position — `get_z_parameters`** (vertical-displacement-event precursor)

| Parameter | Measures | Units |
|---|---|---|
| zcur | Vertical position of the current centroid | m |
| zcur_normalized | zcur normalised by minor radius | 1 |

**Geometry — `get_shape_parameters` / `get_kappa_area`**

| Parameter | Measures | Units |
|---|---|---|
| aminor | Minor radius | m |
| delta | Triangularity (mean of top and bottom) | 1 |
| kappa_area | Elongation from cross-sectional area | 1 |
| squareness | Squareness | 1 |

**Mixed singles** (`get_btor` / `get_h_alpha` / `get_h98` / `get_zeff_parameters` / `get_power_supply_railed`)

| Parameter | Measures | Units |
|---|---|---|
| h98 | H98(y,2) confinement quality factor — degradation is a slow precursor | 1 |
| z_eff | Effective ion charge — impurity content/dilution | 1 |
| h_alpha | H-alpha line emission (uncalibrated) — recycling/edge behaviour | instrument |
| btor | Toroidal magnetic field | T |
| power_supply_railed | E-coil supplies at their limit — Ip feedback authority lost (0/1 flag) | 0 / 1 |

## Label layer — not precursors

| Column | Role | Units |
|---|---|---|
| time_until_disrupt | The label: seconds remaining until disruption (NaN for non-disrupted shots), from the curated d3drdb table | s |
| current_quench_time | Derived indicator: Ip-decay-estimated quench time — disagrees with the label on 3 of 20 shots | s |

## Columns you cannot use (3 of 63)

- **ip_error** — numerically identical to `ip` (the MA-vs-A mismatch makes the subtraction a no-op).
- **dipprog_dt** — MA s⁻¹ while its real-time twin is A s⁻¹; use `dipprog_dt_rt`.
- **time_domain** — constant 1.0 everywhere (flat-top test never fires); carries no information.

Two further cautions from the dictionary: **ip_error_rt** is marked usable but carries a ~205× constant gain (compute `ip_rt − ip_prog_rt` instead), and **radiated_fraction** needs clipping. `ip_prog` is in MA, not A — scale it before differencing against `ip`.

The query script is saved as `precursor_inventory.py` in the working directory.

## Q19 — For a given shot, what is the label and the time, and do the indicators agree?

*label agreement · **partially answerable***

GA asks whether three pipelines agree. We have one — but that one carries **two
independent disruption indicators**, and they disagree on some shots.

In [4]:
%%ask --review
For DIII-D shot 194128, give me the disruption label and the disruption
time. Then tell me whether the dataset contains more than one disruption
indicator, whether they agree, and on which shots they disagree.

I'll consult the d3d-disruption skill (and its Learnings.md, which exists for this skill) before writing the query.

Reading the skill instructions and its learned-error notes.

Applying the learned token fix, I'll write a script that queries the label for 194128 both ways plus the disagreement list.

Running the label query script.

Both queries returned cleanly — marking the work complete and reporting.

## Shot 194128

- **Disruption label** : **disrupted** (per the authoritative curated label)
- **Disruption time (t_disrupt)** : **5.6267 s** (5.626669406890869)

**Coverage note** : the index stops AT the disruption — the last sample for this shot sits at t_disrupt = 5.6267 s exactly, and neither the thermal quench (~0.1–1 ms) nor the current quench (~1–10 ms) is inside this data.

## More than one disruption indicator? Yes — two

| Indicator | Source | 194128 disrupted? | 194128 t_disrupt |
|---|---|---|---|
| `time_until_disrupt` | Human-curated `disruptions` table in d3drdb — **this is the label** | True | 5.626669… s |
| `current_quench_time` | disruption-py's own estimate from the Ip decay — derived indicator, not the label | True | 5.626669… s |

The two are deliberately kept separate in the interface: `disruption_label()` returns the curated one by default, and the derived one requires `derived=True`, so the disrupted count cannot shift silently between definitions.

## Do they agree?

**Yes on shot 194128** — both flag it disrupted at the identical time, 5.6267 s (the index's `t_disrupt` for this shot is a single value; both indicators resolve to it here).

**Across the index: agreement on 17 of 20 shots; disagreement on 3.** The index covers 20 shots (10 curated-disrupted, 10 not — a deliberately balanced constructed slice, not a random sample of DIII-D). On all 3 disagreement shots the pattern is the same direction: the curated table says **not disrupted** , but the Ip-decay estimate says **disrupted** :

| Shot | Curated label | Derived (Ip decay) |
|---|---|---|
| 198244 | not disrupted | disrupted |
| 198643 | not disrupted | disrupted |
| 199062 | not disrupted | disrupted |

**Which to use** : the curated d3drdb label. The Ip-decay-derived indicator is disruption-py's own estimate at commit `9987784e78ef9736e2fe42142683b3458ff38d8d` (v0.15.0.dev0) and is known to disagree with the curated table on exactly these minority of shots — likely a normal ramp-down Ip decay being flagged as a quench, though the index does not record the reason.

The query script is saved as `shot_194128_labels.py` in the working directory.

---
# B. Answerable for one pipeline only

## Q14 — What definition of "disruption" does the pipeline implement?

*definition · **one of three pipelines***

In [5]:
%%ask --review
What definition of a disruption, and of the disruption time, does the
pipeline behind these labels use? Where is that definition documented, and how
would I check it? Be clear about which pipeline this is and which ones are
missing.

The question is about the label-generating pipeline's definition of disruption, so I'll start by planning, then go back to the skill's documentation.

Reading the skill's full documentation to locate the definition of the disruption label.

The SKILL.md describes the two label columns but doesn't state the formal definition. Checking the skill's reference files and learnings.

The learnings flag a token-shadowing issue to pre-empt. Now querying the index for the label columns' full annotations and the provenance record.

Running the query script to pull the label annotations verbatim.

...

Writing a script that probes the pinned commit on GitHub for the label-defining code and lists the d3drdb snapshot's tables.

Running the probe to confirm what's documented and checkable at the pinned commit and in the local snapshot.

The snapshot carries only 5 tables — no curated disruptions table locally. One quick check whether the pinned commit URL resolves on GitHub before writing the report.

Both checks are done — the repo resolves but the pinned commit does not, and the local snapshot lacks the curated table. Closing out the plan and reporting.

## Which pipeline produced these labels

There are two, and the split matters because only one of them actually *defines* the disruption:

1. **disruption-py** (MIT-PSFC), version `0.15.0.dev0`, commit `9987784e78ef9736e2fe42142683b3458ff38d8d`, run by `sammuli` on `omega06.gat.com` on 2026-08-19 — this is the pipeline that produced the NetCDF this index ingests. It computed all 60 physics columns **and one derived disruption indicator** , but it did not decide which shots disrupted.
2. **The human-curated disruption table maintained in DIII-D's shot database (d3drdb)** — this is where the label actually comes from. disruption-py read it and repackaged it as `time_until_disrupt`.

## The definition

**"Disrupted"** — a shot is disrupted if it appears in that human-curated table. This is an *operational* definition: the criterion is whatever the DIII-D operators/analysts who maintain the table applied when they entered the row. **No machine-readable criterion — no Ip threshold, no locked-mode condition, no radiated-fraction cut — is recorded anywhere in this index.** The index's own annotation for the label column says only:

> `time_until_disrupt` — "Seconds remaining until disruption; NaN if the shot did not disrupt" (units s, authoritative)

**"Disruption time"** — `t_disrupt` is the time recorded in that curated table; `time_until_disrupt` is `t_disrupt − t` computed per sample, NaN for non-disrupted shots. The index's samples stop exactly at `t_disrupt`, so the thermal and current quenches (ms-scale, after that instant) are outside the data.

**The derived indicator, for contrast** — `current_quench_time` is disruption-py's own estimate ("Ip-decay-derived quench time", units s, non-authoritative, caveat: "disagrees with the label"). Whatever Ip-decay criterion it applies is implemented in disruption-py's DIII-D method code at the pinned commit — not in this index. It disagrees with the curated label on 3 of the 20 shots (198244, 198643, 199062 — all curated-as-clean, flagged by Ip decay).

## Where the definition is documented

- **In this index** : the parameter annotation layer (`parameter_info('time_until_disrupt')` / `parameter_info('current_quench_time')`) and `disruption_label()`, which reports its source as *"time_until_disrupt (d3drdb, curated)"*. That is the complete machine-readable record — the `method_desc` field for the label group is an **empty string** .
- **In disruption-py** : the Ip-decay logic for `current_quench_time` lives in the DIII-D method modules at commit `9987784…`. I probed the pinned URL recorded in the provenance (`https://github.com/MIT-PSFC/disruption-py/tree/9987784…`) and **the commit no longer resolves on GitHub (HTTP 422) — the repo itself is live, default branch `dev`, but this revision is not currently readable there.** So I cannot quote that code, and you should not treat my description of it as more than the index's one-line annotation.
- **Nowhere accessible here** : the curated table's own entry criteria.

## How you would check it, and what is missing

What you *can* verify in this environment:
- `disruption_label(shot)` — the label and its stated source, per shot.
- `disruption_label(shot, derived=True)` and `label_disagreements()` — the derived indicator and exactly where the two definitions diverge.
- `index_info()` — the pinned commit and run metadata, so any re-derived answer can be tied to a revision.

What is **missing** , plainly:
- **The curated disruption table itself is not in the d3drdb snapshot served here.** I listed its tables: shot records, shot types, signal catalog (2 tables), and shot-summary scalars only. The two dedicated disruption tables were deliberately excluded as legacy, and the summary table's `t_disrupt`-like column is explicitly not a verified label source. So the label's ground truth cannot be independently re-checked in this environment — the index is the only carrier of it.
- **The curators' criterion** — not recorded in any of these systems.
- **The Ip-decay threshold in disruption-py** — lives in source at a commit that currently 422s on GitHub; retrievable only if that revision resurfaces (e.g. a fork or a local clone pinned to `9987784…`).

Bottom line: the label means "the DIII-D curated disruption table says so, at that time," full stop. If your work depends on the precise physical criterion, that has to come from GA's disruption-pipeline documentation or the disruption-py source at the pinned commit — neither is reachable from here today, and the honest position is that the index's one-line annotations are the entire local record.

The check scripts are saved as `definition_labels.py`, `definition_pipeline_check.py`, and `definition_commit_probe.py` in the working directory.

## Q22 — Which shots have every input the pipeline requires?

*input availability · **partially answerable***

In [6]:
%%ask --review
Which parameters does this pipeline produce, and how completely are they
populated across the shots in the index? Identify any parameters that are
missing for a substantial fraction of rows, and say which shots are affected.

I'll start by re-reading the skill documentation and its learned pitfalls, then plan the completeness audit.

Now I'll read the skill's learned-error record to avoid past pitfalls.

The skill documents the exact workflow for this (fill_pct index-wide, per-shot counting, None-handling trap), and the learnings file flags a token pitfall to pre-empt. Planning the audit now.

Inventory complete: 63 columns, 37 below 95% index-wide fill. Now counting per-shot coverage for the sparse groups to identify affected shots.

The per-shot matrix reveals clean patterns (z_eff is all-or-nothing per shot; squareness tracks EFIT where present). One more probe: confirm the 100%-fill columns, verify gap placement at shot tails, and finalize zero-coverage lists.

All three audit scripts ran clean. Marking the plan complete and reporting.

## Parameter inventory and completeness audit

All numbers below come from the index's own annotation layer and direct row counts, at disruption-py `0.15.0.dev0` @ commit `9987784e78ef9736e2fe42142683b3458ff38d8d`. The index holds **63 columns** (including 3 blocked as unusable, marked below) over **3778 rows / 20 shots** (10 disrupted; shots 194000–199687 — a constructed slice, so these fractions describe the slice only). Of the 63, **37 sit below 95% index-wide fill** ; the rest are essentially complete.

### The full parameter list, by producing method, with index-wide fill

| Method (disruption-py) | Parameters | Fill |
|---|---|---|
| get_ip_parameters (5) | ip, ip_prog, dip_dt | 100% |
| get_ip_parameters (blocked) | dipprog_dt; ip_error (94.5%) | 100% / 94.5% |
| get_rt_ip_parameters (5) | ip_rt, ip_prog_rt, dip_dt_rt, dipprog_dt_rt | 100% |
| get_rt_ip_parameters | ip_error_rt | 94.5% |
| get_ohmic_parameters (2) | v_loop | 100% |
| get_ohmic_parameters | p_ohm | 84.6% |
| get_power_parameters (4) | p_nbi, p_ech, p_rad | 100% |
| get_power_parameters | radiated_fraction | 84.6% |
| get_peaking_factors (4) | prad_peaking_cva_rt, prad_peaking_xdiv_rt | 100% |
| get_peaking_factors | ne_peaking_cva_rt, te_peaking_cva_rt | 73.2% |
| get_btor / get_h_alpha / get_h98 / get_zeff_parameters / get_power_supply_railed (5) | btor, h_alpha, power_supply_railed | 100% |
| same group | h98 | 73.7% |
| same group | z_eff | 51.2% |
| get_z_parameters (2) | zcur | 100% |
| get_z_parameters | zcur_normalized | 84.6% |
| get_n1_bradial / get_n1rms (4) | n_equal_1_mode, n_equal_1_normalized | 100% |
| same group | n1rms, n1rms_normalized | 98.8% |
| get_density_parameters (3) | n_e, greenwald_fraction, dn_dt | 94.0% |
| get_rt_density_parameters (3) | n_e_rt, dn_dt_rt | 100% |
| get_rt_density_parameters | greenwald_fraction_rt | 99.9% |
| get_rt_efit_parameters (5) | q95_rt, wmhd_rt, beta_p_rt, li_rt, chisq_rt | 94.2% |
| get_efit_parameters (14) | q95, q0, qstar, wmhd, kappa, chisq, beta_n, beta_p, li, upper_gap, lower_gap | 84.6% |
| get_efit_parameters (derivatives) | dwmhd_dt, dbetap_dt, dli_dt | 82.5% |
| get_shape_parameters / get_kappa_area (4) | aminor, delta, kappa_area | 84.6% |
| same group | squareness | 70.9% |
| Labels and shot phase (3) | time_until_disrupt | 48.8% |
| same group | current_quench_time | 66.4% |
| same group (blocked) | time_domain | 100% |

That enumeration is all 63. Three are blocked regardless of fill (`ip_error`, `dipprog_dt`, `time_domain` — the MA/A unit-defect family); `fetch_samples()` raises on them because they return plausible-but-wrong numbers. The two label columns' shortfalls are **by design** : `time_until_disrupt` is NaN exactly on non-disrupted rows (1845 of 3778 = its 48.8% fill), and `current_quench_time` is filled only where disruption-py's Ip-decay estimator produced a value. I verified the "100%" columns per shot: every one checked is ≥99.6% on its worst shot, so those are genuinely complete.

### Parameters with substantial gaps, and the shots affected

**z_eff — 51.2%, all-or-nothing per shot.** Entirely absent (0 rows) on **10 of 20 shots** : 194000, 194005, 194128, 194292, 194464, 194649, 194968, 195805, 196273, 198244; 100% filled on the other ten. No partial coverage anywhere — the diagnostic either ran for the shot or didn't.

**h98 — 73.7%.** Zero on **4 shots** (194649, 194869, 195985, 198643) and partial on six more, spanning **55.4–88.3%** (195516 lowest at 55.4%; 194464 highest at 88.3%; the others: 194000 at 74.0%, 194968 at 79.5%, 195243 at 71.1%, 195768 at 83.5%). The partial gaps sit late in each discharge.

**ne/te peaking factors — 73.2%.** The pair is zero together on **4 shots** : 194000 (a disrupted shot, t_disrupt 0.8296 s) plus 195805, 196273 and 198244 (non-disrupted) — so three of the four zero-coverage shots are non-disrupted. Partial on 194464, 194869, 194968, 195243, 195516, 195768. The prad peaking twins, by contrast, are 100% filled.

**squareness — 70.9%.** Zero on the **6 shots with the worst EFIT coverage** (194000, 194005, 194128, 194292, 194464, 195516) and partial wherever EFIT is partial — it tracks the EFIT convergence pattern.

**The EFIT block (14 columns + shape/zcur_normalized/p_ohm/radiated_fraction) — 82.5–84.6%.** This is the largest structurally-linked gap. Per-shot q95 coverage: **194464 is the disaster case at 5.8% overall and 0/55 rows in its last 200 ms** ; 195516 (55.4%), 194292 (53.9%), 194128 (73.6%), 194005 (77.1%), 195243 (77.8%), 194000 (67.5%), 194869 (82.9%), 194968 (86.4%), 195768 (83.5%) are all below 90%. No shot has zero EFIT coverage. Critically, **the gaps concentrate at the shot tail — exactly the pre-disruption window** : last-200-ms fill is 0% on 194869 and 195243, 27% on 195516, 29% on 195768, 42% on 194292 (only 194128, at 92.7%, keeps its tail).

**RT-EFIT (94.2%) is the mirror image** : far better filled than offline EFIT everywhere except **194869, where it is entirely zero** (0 rows) while offline EFIT has 82.9% — on that one shot the two measurement paths swap roles. Also below 90% on 194000 (79.2%), 195516 (80.4%), 194968 (87.5%).

**Density (94.0%)** : below 90% on 194000, 194464, 194869, 194968, 195243, 195516 (worst 55.4%). **n1rms (98.8%)** dips on exactly one shot: 198643 (78.1%).

### Practical summary

- For pre-disruption work, **count EFIT columns in your specific window, not index-wide** — 10 of 20 shots lose EFIT coverage exactly in the final 200 ms.
- The most reliable late-shot columns are the current/power/MHD set (ip, dip_dt, v_loop, p_rad, p_nbi, p_ech, h_alpha, btor, n_equal_1_*, the `_rt` current columns, n_e_rt and dn_dt_rt at 100%, prad peaking) — the verified ones are ≥99.6% on every shot, with greenwald_fraction_rt close behind at 99.9% index-wide.
- z_eff cannot support any cross-shot comparison: half the slice simply has none.
- Shot **194464** is the least-covered shot overall (EFIT 5.8%, tail 0%); **194869** is the mirror case (RT-EFIT absent entirely).

The audit scripts are saved as `completeness_inventory.py`, `completeness_by_shot.py`, and `completeness_zero_coverage.py` in the working directory.

## Q25 — Can a hazard-model dataset be assembled?

*survival analysis · **partially answerable***

A hazard model needs, per shot, the time it entered flattop and the time it
disrupted or was censored. The second is available; the first is the open
question.

In [7]:
%%ask --review
I want to build a hazard model of disruptions, which needs for each shot
the time it entered the flattop phase and the time it disrupted or ended
without disrupting. Can this dataset provide both? Give me whichever parts it
can and state plainly what is missing.

I'll plan this out, then consult the relevant skills before writing any code.

Reading the disruption-index skill and its learnings file, plus the relational-db skill for pulse-length metadata.

The skill confirms a critical point: the only phase column (`time_domain`) is the blocked one (flat-top test never fires). Now reading the relational-db skill for pulse-length metadata.

Skill rules noted: the only phase column is the blocked `time_domain`, the curated label is authoritative, and the shot-summary database carries flattop scalars (`ip_flat`, `t_ip_flat`, `ip_flat_duration`, `pulse_length`) for cross-checking. Now I'll write one extraction script that pulls per-shot timing from the index, derives a flattop-entry proxy from the programmed-vs-measured current, and cross-checks against the shot-summary scalars.

Writing the main extraction script for per-shot timing, phase-column check, and flattop-entry proxies.

...

Two findings before finalizing: the three "no flattop" shots fail for different reasons (194869 tracks to 1.3% but the `settled` test breaks — likely a programmed-current overshoot above the sustained flattop level; 194968/195243 have real 6.6–9% tracking offsets), and the official flattop scalars from the shot-summary database are internally inconsistent (e.g. `t_ip_flat` = 878.6 on a 3.02 s pulse, `ip_flat_duration` = 5328 on a 5.68 s pulse — units undocumented, values contradict the waveforms). The pulse-length cross-check, by contrast, is solid: last sample ≈ pulse end within ≤80 ms on all 10 non-disrupted shots.

Since I hit an edit-and-retry cycle (the `None`-row crash, and the offline `ip_prog` scaling surprise), I'll record those lessons in the skill's Learnings file before continuing.

Now finalizing the script with the plateau-median fix and a two-tier flattop estimate (strict tracking-based, plus a program-phase fallback for the two offset shots).

All pieces are computed and cross-checked. Marking the plan complete and summarizing.

Verifying the program-only flattop value for shot 194869 from the saved CSV before writing the corrected report.

## Short answer

Partially. The dataset gives you the **event side of the hazard model directly and verifiably** — disruption times for the disrupted shots, and defensible end times for the rest. It gives you **no flattop-entry time at all as data** : the one column that encodes shot phase is defective and blocked, so I derived a proxy from the programmed-vs-measured plasma current, which works on 18 of 20 shots but is *my construction, not the pipeline's definition*. What is missing, plainly: a pipeline-defined phase boundary, any post-disruption data (both quenches sit outside the index), and a sample you can generalize a hazard rate from.

## What the dataset provides

**Event time for disrupted shots — first-class data.** The curated label gives `t_disrupt` per shot, and I verified sample-by-sample that the **last sample of every disrupted shot sits exactly at `t_disrupt`** (zero offset on all 10). So the event time is exact to the index's own clock.

**End time for non-disrupted shots — last sample, cross-checked.** The index does not carry an explicit "plasma ended" time; the last sample is the available bound. I checked it against the independent pulse-length scalar in the DIII-D shot records: on all 10 non-disrupted shots they agree to **≤ 80 ms** (most within 30 ms), so `t_last_sample` is a sound censoring time.

**Per-shot sampling coverage** — first sample is 0.100 s on all 20 shots; cadence is non-uniform on disrupted shots (coarse through the bulk, fine near the disruption), so derive spacing from the `time` values rather than assuming a rate.

## What is missing, plainly

- **No usable flattop marker.** The only phase column (`time_domain`: 1 ramp-up / 2 flat-top / 3 ramp-down) is one of the three blocked parameters — the MA/A unit defect holds its threshold test permanently false, so it never leaves its first value. It returns plausible-looking numbers, which is why the interface refuses to serve it.
- **No post-disruption data.** The index stops at `t_disrupt`; the thermal quench (~0.1–1 ms) and current quench (~1–10 ms) that follow are outside it. Fine for a "time-to-disruption" hazard definition, unavailable if you want quench-resolved outcomes.
- **The shot set is a constructed 10/10 slice** (20 shots, 194000–199687), deliberately balanced — not a random sample of DIII-D. Within-slice hazard modelling is legitimate; an archive-wide disruption rate is not computable from it.
- The shot-summary database's official flattop scalars are **internally inconsistent and unusable** — e.g. `t_ip_flat` = 878.648 on a 3.02 s pulse, `ip_flat_duration` = 5328.4 on a 5.68 s pulse (units undocumented; values contradict the waveforms). Do not substitute them.

## The derived flattop-entry proxy (18/20 shots)

Method: from the real-time current pair (both in amperes; they track within a few percent — the offline pair cannot be used, its programmed column carries a ~2× scaling defect), take a robust plateau level (median of the top decile of |programmed current|), and call flattop entered when the programmed current holds ≥95% of plateau **and** the measured current tracks it within 5% for 10 consecutive samples. Where tracking fails but the program settles, a looser program-only tier is reported.

| Shot | Disrupted | t_flattop (s) | Tier | End/disrupt (s) | At-risk from flattop (s) |
|---|---|---|---|---|---|
| 194000 | yes | 0.734 | measured-tracking | 0.830 | 0.096 |
| 194005 | no | 0.950 | measured-tracking | 3.025 | 2.075 |
| 194128 | yes | 0.700 | measured-tracking | 5.627 | 4.927 |
| 194292 | yes | 0.975 | measured-tracking | 4.332 | 3.357 |
| 194464 | yes | 0.975 | measured-tracking | 3.184 | 2.209 |
| 194649 | yes | 0.650 | measured-tracking | 3.352 | 2.702 |
| 194869 | no | 0.775 | program-only | 3.000 | 2.225 |
| 194968 | yes | — | none | 1.121 | — |
| 195243 | no | — | none | 1.200 | — |
| 195516 | yes | 0.775 | measured-tracking | 1.218 | 0.443 |
| 195768 | yes | 0.675 | measured-tracking | 5.297 | 4.622 |
| 195805 | no | 0.925 | measured-tracking | 6.925 | 6.000 |
| 195985 | yes | 0.825 | measured-tracking | 4.800 | 3.975 |
| 196273 | no | 0.925 | measured-tracking | 6.225 | 5.300 |
| 196347 | yes | 1.625 | measured-tracking | 5.476 | 3.851 |
| 198244 | no | 0.925 | measured-tracking | 5.250 | 4.325 |
| 198643 | no | 0.875 | measured-tracking | 5.100 | 4.225 |
| 199062 | no | 1.875 | measured-tracking | 6.425 | 4.550 |
| 199359 | no | 1.175 | measured-tracking | 5.925 | 4.750 |
| 199687 | no | 2.425 | measured-tracking | 5.975 | 3.550 |

On the two shots with no estimate at all (194968, 195243 — both short, ~1.1–1.2 s), the measured current sits a real 6.6–9% below the programmed level for the whole discharge; there is no interval that satisfies both criteria, and I would rather report nothing than invent a boundary. Shot 194869 is the one program-only case: the measured current tracks the programmed plateau to within ~1.3%, just above my 5% gate's edge behaviour around the settling transient, so its entry time rests on the program settling alone — its programmed ramp-down (for reference) begins at 1.150 s. Treat all proxy values as **analyst-derived with chosen thresholds (95% of plateau, 5% tracking, 10-sample hold)** — defensible, reproducible from the saved script, but not a pipeline-defined phase.

The full table — including disruption label, `t_disrupt`, first/last sample, programmed ramp-down start, and both at-risk durations — is saved as **`hazard_model_inputs.csv`** in the working directory (scripts: `hazard_timing.py`, `hazard_flattop_proxy.py`, `hazard_diagnostics.py`, `hazard_debug_proxy.py`).

---
# C. Correctly refused — and the refusal is the finding

## Q17 — What fraction of a campaign disrupted?

*rate · **correctly refused***

Two independent blockers: the index is a constructed slice rather than a
sample, and GA's question excludes *intentional* disruptions, which nothing in
the data flags.

In [8]:
%%ask --review
What fraction of DIII-D shots in the 2023–2024 campaigns disrupted,
excluding intentional disruptions?

I'll plan this archive-wide rate question, which needs the relational DB rather than the 20-shot index.

Reading the relational DB skill instructions first.

Reading the last few lines of the skill file.

Checking prior scripts and their recorded findings before querying fresh.

Reading the earlier schema-listing script to reuse its table-enumeration approach.

...

The BRIEF query needs `RUN` since `time_of_shot` lives in SUMMARIES — fixing and re-running.

The probe results are conclusive. Before reporting, I'm recording the SQL-dialect lessons from this session's edit-and-retry cycles in the skill's Learnings file.

Reading the earlier index-query script to reuse its exact API pattern for the composition recount.

Now recomputing the index label composition directly, so any figure I state is traceable to a query run in this session.

## Answer: this fraction cannot be computed from the data available in this environment — and I'd rather say that plainly than hand you a number that looks like one.

Here is everything the question decomposes into, and what I found for each part.

**The denominator exists.** The DIII-D shot records cover the 2023–2024 campaigns well: **5,055 plasma shots** fall in the window by shot timestamp (2,278 in 2023, 2,777 in 2024; the session-date field agrees at 2,279 + 2,777), spanning shot numbers 193503–202230. Timestamps are populated for the whole window, so the denominator is solid.

**The numerator does not exist here.** Every candidate source for "which of those 5,055 disrupted" fails:

- **The authoritative label store** — the disruption index built from disruption-py — covers only **20 shots** , all inside your window (2023-01-23 to 2024-07-22). Its label composition, which I queried directly, is **10 disrupted and 10 non-disrupted** , and the index's own metadata flags it `is_representative_sample: False`. So the 50% you would read off those 20 shots is a property of how the index slice was assembled, not of the campaigns — it cannot be scaled up to the 5,055.
- **The dedicated disruption-label tables** from the original database are legacy and deliberately excluded from the snapshot served here (per GA, do not use).
- **The shot-summary disruption scalars** are empty for this entire window — disruption time filled on 0 of 5,055 shots — and even where populated they are explicitly not a verified label source.
- **Fault codes** are blank for all 5,055 shots in the window.

**The intent exclusion cannot be applied at all.** There is no field anywhere in this snapshot that distinguishes an intentional disruption (a deliberately triggered shot, e.g. for mitigation studies) from an unintentional one. The shot-quality flag is NULL on every row; the shot-type categories carry no intent meaning (23,328 are simply "unknown"); and a free-text search of the operator descriptions for "disrupt" returns zero hits in the window. So even with a numerator, this dataset could not separate the two populations.

## What it would take

A real answer needs disruption-py (or GA's disruption-pipeline labels) run across the full 2023–2024 shot set, plus GA's classification of intentional disruptions. Neither is served in this environment; the honest statement is that the fraction is **not computable here** , not that it is low or high.

The query scripts are saved as `campaign_disrupt_probe.py`, `campaign_disrupt_denominator.py`, and `campaign_index_composition.py` in the working directory.

## Q18 — Which discharge phase was each disruption in?

*phase · **blocked, and the reason is instructive***

The dataset contains a column intended to answer exactly this. It does not
work — and how the agent handles that is the point of the question.

In [9]:
%%ask --review
For the disrupted shots in the index, which phase of the discharge was
each disruption in — ramp-up, flattop, or ramp-down?

Reading the disruption skill instructions and its learned pitfalls before writing code.

The skill file is loaded; now reading its Learnings.md for known pitfalls.

Learnings loaded. Now checking the working directory for the hazard-model outputs I can reuse.

Reading the final flattop-proxy script so I can reuse its exact (debugged) logic.

The saved CSV holds the v4 outputs; reading it to cross-check the new script's results.

...

Running the boundary diagnostic.

The diagnostics settle the one ambiguous shot: 194968's program overshot ~10%, never held a flattop, and was descending for its last 0.35 s — the strict "no-entry ⇒ ramp-up" rule mislabels it. Fixing the classifier to test the program's slope at t_dis.

Re-running the classifier with the corrected rule.

Before the final report, recording the phase-classification lesson in the skill's Learnings file, as required after the edit-and-retry cycle.

## Phase at disruption — all 10 disrupted shots in the index

Of the 10 disrupted shots (out of 20 in this index, disruption-py `0.15.0.dev0` @ `9987784e…`): **6 disrupted in flattop, 4 during ramp-down, 0 during ramp-up.** The dataset itself has no phase column you can use — the pipeline's own `time_domain` is one of the three blocked parameters (its threshold test never fires) — so every boundary below is **analyst-derived from the current waveform** , using the same audited method as the hazard inputs: flattop entered when the programmed current (real-time pair, both in amperes) holds ≥95% of its plateau with the measured current tracking within 5% for 10 consecutive samples; ramp-down starts when the program first falls below 90% of plateau. The index stops at `t_disrupt`, so no quenches are included.

| Shot | t_disrupt (s) | Phase at disruption | Evidence |
|---|---|---|---|
| 194000 | 0.830 | flattop | entered 0.734 s; program still 100% of plateau at t_dis |
| 194292 | 4.332 | flattop | entered 0.975 s; program 99.6%, Ip 96.6% |
| 194464 | 3.184 | flattop | entered 0.975 s; program 100%, Ip 95.9% |
| 194649 | 3.352 | flattop | entered 0.650 s; program 100%, Ip 97.0% |
| 195985 | 4.800 | flattop | entered 0.825 s; program 100%, Ip 65.9% — Ip already collapsing while the program held |
| 196347 | 5.476 | flattop | entered 1.625 s; program 100%, Ip 42.8% — same pattern, more advanced |
| 194128 | 5.627 | ramp-down | ramp-down began 5.400 s; disrupted 227 ms into it, at 76% of plateau |
| 195516 | 1.218 | ramp-down | ramp-down began 1.122 s; disrupted 96 ms into it, at 80% |
| 195768 | 5.297 | ramp-down | ramp-down began 5.125 s; disrupted 172 ms into it, at 79% |
| 194968 | 1.121 | ramp-down (no flattop established) | program peaked at 0.775 s (overshooting ~10%), never held a flattop, and was descending for its final 0.35 s; disrupted at 68% of peak program, Ip at 42% |

Two patterns worth noting in the evidence column. The three clean ramp-down cases (194128, 195516, 195768) show **measured Ip tracking the programmed ramp closely into the disruption** (e.g. 194128: Ip 75.7% vs program 76.3% at t_dis) — the plasma was following the commanded ramp when it broke. By contrast 195985 and 196347 are **flattop disruptions with the current already collapsing** : the program held 100% but measured Ip sat at 66% and 43% of plateau at the final sample — the plasma was losing current against a flat command. 194968 is the odd case — an overshoot-then-descend program that never established flattop at all; a strict "no-flattop ⇒ ramp-up" rule would mislabel it, so I classified it by the program's slope at t_disrupt (descending, well below its own peak).

**Caveats, plainly** : the phase boundaries are my construction from the RT current pair with the stated thresholds — reproducible from `disruption_phase.py`, but not a pipeline-defined phase; boundary cases within ~25 ms (the coarse cadence) are not resolvable; and the counts describe this constructed 10-disrupted-shot slice only, not DIII-D disruption statistics. Full per-shot table with entry times, ramp-down starts, margins and Ip fractions: `disruption_phase.csv` (scripts: `disruption_phase.py`, `disruption_phase_diag.py`).

---
# D. What would unblock the rest

## Summary

*synthesis · **answerable***

In [10]:
%%ask --review
Across everything in this notebook: which of GA's disruption competency
questions can this index answer today, which need something more, and what
specifically is missing in each case? Be concrete about what would have to
arrive for each blocked question to become answerable.

## Competency assessment: this index vs. the questions this notebook asked

The object under test is a **20-shot constructed slice** (shots 194000–199687, 10 disrupted / 10 not, `is_representative_sample: False`) built by disruption-py `0.15.0.dev0` @ `9987784e78ef9736e2fe42142683b3458ff38d8d`, with labels repackaged from the human-curated d3drdb disruption table. Everything below classifies each question this notebook has asked against exactly that object.

### ✅ Answerable today, from the index alone

| Question | What the index delivers |
|---|---|
| **Label provenance** | Full lineage: code, version, commit, run date, operator, host, input shot range, ingest record |
| **What each parameter measures + units** | Complete 63-column dictionary with method attribution, units, and defect annotations |
| **Per-shot labels and disruption times** | Curated label + exact `t_disrupt` per shot (verified: last sample sits at `t_disrupt` on all 10 disrupted shots) |
| **Multiple indicators & their agreement** | Two indicators identified (curated vs Ip-decay-derived), 17/20 agreement, the 3 disagreement shots named |
| **Parameter completeness** | Full per-shot fill audit; every parameter with substantial gaps identified with its affected shots |
| **Pre-disruption plasma state** | 60 physics columns as time series up to `t_disrupt` — precursor state at any time-before-disruption is queryable |

Even these answers carry the index's known defects: 3 parameters blocked outright (`ip_error`, `dipprog_dt`, `time_domain` — the MA/A unit family), `ip_error_rt` carrying a ~205× gain, and late-time EFIT gaps on 10 of 20 shots sitting exactly on the pre-disruption window.

### ⚠️ Answerable only with analyst construction (data lacks the field; a proxy exists)

| Question | Status | What would make it first-class |
|---|---|---|
| **Flattop-entry times** (hazard model) | Proxy from the RT current pair for 18/20 shots; none for 194968, 195243 | A working `time_domain` (fix the unit mismatch so the threshold test can fire, re-run the pipeline) **or** consistent per-shot phase-boundary times from GA — the served `t_ip_flat`-family scalars are internally contradictory and unusable |
| **Phase at disruption** | Derived: 6 flattop, 4 ramp-down, 0 ramp-up | Same fix; alternatively an operator-recorded phase per shot for the two unresolvable cases |
| **Hazard-model event/censoring times** | Event times exact; censoring cross-checked to ≤80 ms vs pulse length | Nothing missing — this half is genuinely delivered |

### ❌ Blocked — and the specific artifact each one needs

**Campaign-scale disruption rate ("what fraction of 2023–24 shots disrupted")** — the denominator exists (5,055 plasma shots), the numerator does not. Would become answerable if **either** of these arrived:
- disruption-py run archive-wide over 2023–24 (or GA's disruption-label table actually served in the d3drdb snapshot — the dedicated tables were excluded as legacy, and the summary scalars are empty for the whole window).

**Excluding intentional disruptions** — no path exists in this snapshot at all. Requires a **per-shot intent flag** (intentional vs unintentional disruption). Fault codes are blank, shot-quality is NULL on every row, and free-text descriptions contain zero disruption references. Without this field the exclusion is unimplementable even given labels.

**The physical definition behind the curated label** — the label is operational ("the curated table says so"). Requires GA's documentation of the curators' entry criteria. The Ip-decay criterion behind the derived indicator additionally requires the disruption-py source **at the pinned commit** — that commit currently returns HTTP 422 on GitHub; a tag, fork, or local clone pinned to `9987784e…` would unblock it.

**Quench-resolved analysis** (thermal quench ~0.1–1 ms, current quench ~1–10 ms) — both sit after `t_disrupt`, and the index stops there by construction. Requires a re-run without the stop-at-disruption truncation, or a raw MDSplus fetch (d3d-shot-fetcher / toksearch-mds) for the post-event window.

**Generalizable warning models / training** — 20 deliberately balanced shots support method development and within-slice statistics only. Requires an archive-wide, representative ingest (`is_representative_sample: True`).

**Cross-shot work in specific parameters** — `z_eff` is absent on 10 of 20 shots (needs the underlying diagnostic processed for those shots); EFIT is near-absent late-time on 10 shots, catastrophically on **194464** (5.8% overall, 0% in its last 200 ms); **194869** has RT-EFIT entirely missing while offline EFIT survives — the two paths swap, so neither set is complete alone.

### The shopping list, condensed

Five arrivals would unblock nearly everything: **(1)** archive-wide disruption labels for 2023–24, **(2)** a per-shot intentional-disruption flag, **(3)** the unit fix that revives `time_domain` (or served phase boundaries), **(4)** disruption-py source at the pinned commit plus GA's curation criteria, **(5)** sampling extended past `t_disrupt`. Items 1+2 alone convert the blocked campaign-rate question into a one-line query; item 3 converts both analyst-proxy answers into pipeline-delivered data.

The reproducible artifacts backing the proxy-tier answers are saved in the working directory (`hazard_model_inputs.csv`, `disruption_phase.csv`, and the query scripts listed in earlier sections).